# S-GMS

Monte-Carlo dropout voting over the candidate diagrams, then selection of the
representative records.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nets.MobileNetV3_m import build_classifier
from nets.selectwave import select_group, select_all, split_pools
from utils.dataloader import make_predict_dataset, list_images
from utils.utils_fit import threshold_curve
from utils.utils import read_table

RDTD_DIR   = 'data/rdtd_labeled/'
WEIGHTS    = 'runs/final.weights.h5'
THRESHOLD  = None        # None to adopt the Youden-optimal threshold
CURVE_DIR  = 'data/midr/'
TARGET_DIR = 'data/target/'
OUT        = 'output/'

CHANNELS = 3
PASSES   = 30     # forward passes per pool
TOP_N    = 2      # records retained per class

In [ ]:
paths, _ = list_images(RDTD_DIR)
ds = make_predict_dataset(paths, size=(224, 224))

model = build_classifier(CHANNELS, 1, 128, 0.2)
if os.path.exists(WEIGHTS):
    model.load_weights(WEIGHTS)

In [ ]:
def curve_of(path):
    """Response curve belonging to one record."""
    key = os.path.splitext(os.path.basename(str(path)))[0]
    hit = glob.glob(os.path.join(CURVE_DIR, key + '*.txt'))
    return np.loadtxt(hit[0]) if hit else None

def target_of(group):
    """Target quantiles of one group."""
    lo = glob.glob(os.path.join(TARGET_DIR, group + '*16*.txt'))
    hi = glob.glob(os.path.join(TARGET_DIR, group + '*84*.txt'))
    if not (lo and hi):
        return None, None
    return np.loadtxt(lo[0]), np.loadtxt(hi[0])

In [ ]:
group = None    # restrict to one group, or None for all records
subset = paths if group is None else [p for p in paths if group in p]

rec = select_group(model, ds, subset, curve_of,
                   passes=PASSES, top_n=TOP_N)

for side in ('pos', 'neg'):
    r = rec[side]
    print(f"{side}: pool {r['n_pool']}  selected {len(r['paths'])}  "
          f"votes {r['freq']}/{PASSES}")

In [ ]:
t_lo, t_hi = target_of(group) if group else (None, None)

fig, ax = plt.subplots(figsize=(6, 4.5))
for key, colour in (('pos', '#c0392b'), ('neg', '#2c6fbb')):
    c = rec[key]['curve']
    if c is not None:
        ax.plot(c, color=colour, lw=1.8, label=f'{key} mean')
for t, c, ls in ((t_lo, '#c0392b', '--'), (t_hi, '#2c6fbb', '--')):
    if t is not None:
        ax.plot(t, color=c, ls=ls, lw=1.0)

ax.set_xlabel('floor'); ax.set_ylabel('MIDR (%)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'group_curves.png'), dpi=200)

In [ ]:
table = select_all(model, ds, paths, curve_of, target_of,
                   passes=PASSES, top_n=TOP_N, out_dir=OUT)
table.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.violinplot([table['drc'].dropna().values], showmeans=True, showmedians=True)
ax.set_ylabel('DRC (%)'); ax.set_xticks([1])
ax.set_xticklabels([f'CT-GMS ({TOP_N} per class)'])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'drc_distribution.png'), dpi=200)